# Path 3 — Market reactions & event study

Join **macro surprises** with **forward returns** and optional **vol tags**.
Requires: bootstrapped DB, `vol_indices` (bootstrap or `python -m mini_hedge.cli fetch-vol`), and yfinance for SPY/TLT.


In [ ]:
import os, sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "mini_hedge").is_dir():
    project_root = _cwd
elif (_cwd.parent / "mini_hedge").is_dir():
    project_root = _cwd.parent
else:
    _ex = (os.environ.get("MINI_HEDGE_ROOT") or "").strip()
    project_root = Path(_ex).resolve() if _ex else None
    if project_root is None or not (project_root / "mini_hedge").is_dir():
        raise RuntimeError("Set cwd to repo root or notebooks/, or MINI_HEDGE_ROOT")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from mini_hedge.surprises import compute_surprises
from mini_hedge.prices import fetch_yfinance, CPI_RELEASE_DATES
from mini_hedge.event_study import forward_close_returns, tag_events_with_vol

print("Imports Successful")

In [ ]:
# Build a minimal CPI event table (release dates × surprise z where available)
surp = compute_surprises(method="ema")
releases = pd.DataFrame({"event_date": pd.to_datetime(CPI_RELEASE_DATES)})
# nearest prior month surprise row per release (simplified join on calendar month)
surp["ym"] = surp["date"].dt.to_period("M")
releases["ym"] = (releases["event_date"] - pd.DateOffset(months=1)).dt.to_period("M")
ev = releases.merge(surp[["ym", "surprise", "surprise_zscore", "signal"]], on="ym", how="left")
ev = ev.drop(columns=["ym"]).dropna(subset=["surprise"])
ev.head(5)

In [ ]:
spy = fetch_yfinance("SPY", start="1993-01-29")
tlt = fetch_yfinance("TLT", start="2002-07-26")
ret = forward_close_returns(spy, ev.tail(80), event_date_col="event_date", horizons=(0, 1, 5))
ret[["event_date", "surprise_zscore", "ret_h0", "ret_h1", "ret_h5"]].describe()

In [ ]:
tagged = tag_events_with_vol(ev.tail(40), spy_prices=spy, tlt_prices=tlt)
tagged[["event_date", "vix_pre", "move_pre", "rv_spy_h0_h5_ann", "vol_miss_spy"]].tail(8)